# 04 — Synthetic ground-truth Yamada recovery

End-to-end benchmark:

\[
G_{\rm true}\rightarrow G_\lambda\rightarrow V_{\lambda,N}\rightarrow\widehat G
\rightarrow\Upsilon(\widehat G).
\]

Only orientation-preserving invertible affine maps \(F(x)=Mx+b\), \(\det M>0\), are used for the deformation step, so \(G_\lambda\) is ambient-isotopic to \(G_{\rm true}\). The headline metric is simply
\(\Upsilon(\widehat G)=\Upsilon(G_{\rm true})\).

The suite includes knots/links, theta graphs, and bridgeless trivalent graphs. High-valence handlebody spines are intentionally avoided because a thickened handlebody need not select a unique spine.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "knotted_graph").exists():
    raise RuntimeError("Run this notebook from inside the KnottedGraph checkout.")
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp
from skimage.morphology import ball, dilation, skeletonize

import knotted_graph
from knotted_graph.core import simplify_edges
from knotted_graph.extraction import skeleton_image_to_graph
from knotted_graph.projection import compute_yamada_polynomial

kg_path = Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents:
    raise RuntimeError(f"A stale knotted_graph was imported from {kg_path}")

A = sp.Symbol("A")
BOUND = 1.35
RESOLUTIONS = list(range(150, 301, 25))   # 150, 175, ..., 300
TUBE_RADII_VOX = [1, 2, 3]
TRANSFORMS = ["identity", "rotate", "affine"]
PROJECTION_SAMPLES = 12

RESUME = True
CHECKPOINT = ROOT / "User_guide" / "benchmarks" / "synthetic_ground_truth_results.jsonl"
print("branch-local KnottedGraph:", kg_path)
print("checkpoint:", CHECKPOINT)


## Ground-truth suite

The default large suite contains 13 distinct spatial-graph examples:

`unknot`, `trefoil`, `cinquefoil`, `figure8`, `unlink2`, `hopf_link`,
`theta3_planar`, `theta3_twisted`, `K4`, `triangular_prism`, `cube`,
`K3_3`, and `petersen`.

The last five are bridgeless trivalent graphs. Planar graphs use deterministic planar layouts; non-planar graphs use deterministic generic 3D embeddings with a positive nonincident-edge clearance.


In [ ]:
@dataclass
class Case:
    name: str
    graph: nx.MultiGraph
    radius_cap: float


def emb(nodes, edges):
    G = nx.MultiGraph()
    for n, p in nodes.items():
        G.add_node(n, pos=np.asarray(p, float))
    for u, v, p in edges:
        G.add_edge(u, v, pts=np.asarray(p, float))
    return G


def norm(P, scale=.72):
    P = np.asarray(P, float)
    P -= P.mean(0)
    return P * (scale / np.max(np.linalg.norm(P, axis=1)))


def loop_case(name, P, cap=.055):
    P = norm(P)
    P[-1] = P[0]
    return Case(name, emb({0:P[0]}, [(0,0,P)]), cap)


def unknot(n=700):
    t=np.linspace(0,2*np.pi,n)
    return loop_case("unknot", np.c_[np.cos(t),np.sin(t),0*t], .15)


def torus(name,p,q,n=1200):
    t=np.linspace(0,2*np.pi,n)
    r=1+.35*np.cos(q*t)
    P=np.c_[r*np.cos(p*t),r*np.sin(p*t),.35*np.sin(q*t)]
    return loop_case(name,P,.050)


def figure8(n=1200):
    t=np.linspace(0,2*np.pi,n)
    P=np.c_[(2+np.cos(2*t))*np.cos(3*t),
            (2+np.cos(2*t))*np.sin(3*t),
            np.sin(4*t)]
    return loop_case("figure8",P,.045)


def two_loops(name,linked,n=700):
    t=np.linspace(0,2*np.pi,n)
    if linked:
        c1=np.c_[np.cos(t),np.sin(t),0*t]
        c2=np.c_[1+np.cos(t),0*t,np.sin(t)]
        P=norm(np.vstack([c1,c2]),.74); c1,c2=P[:n],P[n:]; cap=.045
    else:
        c1=np.c_[.42*np.cos(t)-.55,.42*np.sin(t),0*t]
        c2=np.c_[.42*np.cos(t)+.55,.42*np.sin(t),0*t]; cap=.07
    c1[-1]=c1[0]; c2[-1]=c2[0]
    return Case(name,emb({0:c1[0],1:c2[0]},[(0,0,c1),(1,1,c2)]),cap)


def theta(name,twisted=False,n=650):
    t=np.linspace(0,1,n); x=-.72+1.44*t
    if not twisted:
        C=[np.c_[x,a*np.sin(np.pi*t),0*t] for a in (-.58,0,.58)]
    else:
        C=[np.c_[x,-.52*np.cos(np.pi*t), .24*np.sin(np.pi*t)],
           np.c_[x, .52*np.cos(np.pi*t),-.24*np.sin(np.pi*t)],
           np.c_[x, .72*np.sin(np.pi*t),0*t]]
        for P in C: P[0]=[-.72,0,0]; P[-1]=[.72,0,0]
    return Case(name,emb({"u":C[0][0],"v":C[0][-1]},
                         [("u","v",P) for P in C]),.065)


def segdist(p1,q1,p2,q2):
    u=q1-p1; v=q2-p2; w=p1-p2
    a=u@u; b=u@v; c=v@v; d=u@w; e=v@w; D=a*c-b*b
    if D<1e-14: s=0.; t=np.clip(e/c if c>1e-14 else 0.,0,1)
    else: s=np.clip((b*e-c*d)/D,0,1); t=np.clip((a*e-b*d)/D,0,1)
    if a>1e-14: s=np.clip((b*t-d)/a,0,1)
    if c>1e-14: t=np.clip((b*s+e)/c,0,1)
    return np.linalg.norm(w+s*u-t*v)


def clearance(G,P):
    E=list(G.edges()); best=np.inf
    for i,(u,v) in enumerate(E):
        for a,b in E[i+1:]:
            if {u,v}&{a,b}: continue
            best=min(best,segdist(P[u],P[v],P[a],P[b]))
    return best


def cubic_case(name,G,planar,seed,cap=.040):
    G=nx.Graph(G)
    assert all(d==3 for _,d in G.degree()) and not list(nx.bridges(G))
    if planar:
        ok,_=nx.check_planarity(G); assert ok
        p=nx.planar_layout(G)
        X=norm(np.array([[p[n][0],p[n][1],0.] for n in G]),.72)
        P={n:X[i] for i,n in enumerate(G)}
    else:
        for trial in range(100):
            p=nx.spring_layout(G,dim=3,seed=seed+trial,iterations=300)
            X=norm(np.array([p[n] for n in G]),.72)
            P={n:X[i] for i,n in enumerate(G)}
            if clearance(G,P)>.055: break
        else: raise RuntimeError(f"no clear embedding for {name}")
    return Case(name,emb(P,[(u,v,np.linspace(P[u],P[v],100)) for u,v in G.edges()]),cap)


CASES=[
    unknot(), torus("trefoil",2,3), torus("cinquefoil",2,5), figure8(),
    two_loops("unlink2",False), two_loops("hopf_link",True),
    theta("theta3_planar"), theta("theta3_twisted",True),
    cubic_case("K4",nx.complete_graph(4),True,11,.055),
    cubic_case("triangular_prism",nx.circular_ladder_graph(3),True,12),
    cubic_case("cube",nx.cubical_graph(),True,13),
    cubic_case("K3_3",nx.complete_bipartite_graph(3,3),False,14,.032),
    cubic_case("petersen",nx.petersen_graph(),False,15,.030),
]

[(c.name,c.graph.number_of_nodes(),c.graph.number_of_edges()) for c in CASES]


In [ ]:
def Rxyz(a,b,c):
    a,b,c=np.deg2rad([a,b,c])
    Rx=np.array([[1,0,0],[0,np.cos(a),-np.sin(a)],[0,np.sin(a),np.cos(a)]])
    Ry=np.array([[np.cos(b),0,np.sin(b)],[0,1,0],[-np.sin(b),0,np.cos(b)]])
    Rz=np.array([[np.cos(c),-np.sin(c),0],[np.sin(c),np.cos(c),0],[0,0,1]])
    return Rz@Ry@Rx


def affine(name):
    if name=="identity": M,b=np.eye(3),np.zeros(3)
    elif name=="rotate": M,b=Rxyz(21,34,13),np.array([.04,-.03,.02])
    else:
        M=Rxyz(17,-23,31)@np.diag([1.08,.91,1.03])@np.array([[1,.13,0],[0,1,.09],[.05,0,1]])
        b=np.array([-.03,.04,-.02])
    assert np.linalg.det(M)>0
    return M,b


def deform(G,name):
    M,b=affine(name); H=nx.MultiGraph()
    for n,d in G.nodes(data=True): H.add_node(n,pos=d["pos"]@M.T+b)
    for u,v,k,d in G.edges(keys=True,data=True): H.add_edge(u,v,pts=d["pts"]@M.T+b)
    return H


def interior_sep(G,trim=.15):
    E=[(u,v,np.asarray(d["pts"])) for u,v,k,d in G.edges(keys=True,data=True)]
    if len(E)<2:return np.inf
    best=np.inf
    for i,(u,v,P) in enumerate(E):
        for a,b,Q in E[i+1:]:
            if {u,v}&{a,b}:
                m=max(1,int(trim*len(P))); n=max(1,int(trim*len(Q)))
                P=P[m:-m] if 2*m<len(P) else P
                Q=Q[n:-n] if 2*n<len(Q) else Q
            for s in range(0,len(P),128):
                best=min(best,float(np.sqrt(np.sum((P[s:s+128,None]-Q[None])**2,-1).min())))
    return best


def admissible(case,G,N,r):
    dx=2*BOUND/(N-1); rw=r*dx; sep=interior_sep(G)
    limit=min(case.radius_cap,np.inf if not np.isfinite(sep) else .4*sep)
    return rw<=limit,rw,sep,limit


def resample(P,step):
    out=[]
    for p,q in zip(P[:-1],P[1:]):
        n=max(2,int(np.ceil(np.linalg.norm(q-p)/step))+1)
        out.append(np.linspace(p,q,n,endpoint=False))
    out.append(P[-1:])
    return np.vstack(out)


def voxelize(G,N,r):
    V=np.zeros((N,N,N),bool); dx=2*BOUND/(N-1)
    for u,v,k,d in G.edges(keys=True,data=True):
        P=resample(d["pts"],dx/3)
        I=np.rint((P+BOUND)/(2*BOUND)*(N-1)).astype(int)
        I=np.clip(I,0,N-1); V[I[:,0],I[:,1],I[:,2]]=1
    return dilation(V,footprint=ball(r))


def world(G,N):
    H=nx.MultiGraph(G); dx=2*BOUND/(N-1); o=np.array([-BOUND]*3)
    for n,d in H.nodes(data=True): d["pos"]=o+dx*np.asarray(d["pos"],float)
    for u,v,k,d in H.edges(keys=True,data=True): d["pts"]=o+dx*np.asarray(d["pts"],float)
    return H


def recover(V,N):
    S=skeletonize(V,method="lee")
    return simplify_edges(world(skeleton_image_to_graph(S),N))


def yamada(G):
    R=compute_yamada_polynomial(
        G,A,rotation_angles=None,num_rotation_samples=PROJECTION_SAMPLES,
        normalize=True,n_jobs=1,method="recursive",return_result=True)
    return sp.expand(R.polynomial)


def same(x,y):
    return sp.simplify(sp.together(sp.expand(x-y)))==0


## Graph-level certification

Before voxelization, the notebook checks that every allowed affine deformation gives the same normalized Yamada polynomial as its ground truth. Any failure here aborts the benchmark rather than contaminating the recovery statistics.


In [ ]:
targets={c.name:yamada(c.graph) for c in CASES}
for c in CASES:
    for t in TRANSFORMS:
        assert same(yamada(deform(c.graph,t)),targets[c.name]), (c.name,t)
print("PASS: all affine deformations preserve graph-level Yamada.")


In [ ]:
def key(c,t,N,r): return (c,t,int(N),int(r))

done={}
if RESUME and CHECKPOINT.exists():
    for line in CHECKPOINT.read_text().splitlines():
        if line.strip():
            row=json.loads(line); done[key(row["case"],row["transform"],row["resolution"],row["radius_vox"])]=row
    print("Resuming from",len(done),"completed points.")

records=[]
CHECKPOINT.parent.mkdir(parents=True,exist_ok=True)

for c in CASES:
    for t in TRANSFORMS:
        G=deform(c.graph,t)
        for N in RESOLUTIONS:
            for r in TUBE_RADII_VOX:
                K=key(c.name,t,N,r)
                if K in done:
                    records.append(done[K]); print("CACHED",K); continue
                ok,rw,sep,lim=admissible(c,G,N,r)
                row=dict(case=c.name,transform=t,resolution=N,radius_vox=r,
                         radius_world=rw,separation=sep,limit=lim,
                         admissible=bool(ok),success=None,final_V=None,final_E=None,error=None)
                if ok:
                    try:
                        H=recover(voxelize(G,N,r),N)
                        row.update(success=bool(same(yamada(H),targets[c.name])),
                                   final_V=H.number_of_nodes(),final_E=H.number_of_edges())
                    except Exception as e:
                        row.update(success=False,error=f"{type(e).__name__}: {e}")
                records.append(row)
                with CHECKPOINT.open("a") as f:f.write(json.dumps(row)+"\n")
                mark="SKIP" if not ok else ("PASS" if row["success"] else "FAIL")
                print(f"{mark:4s} {c.name:18s} {t:8s} N={N} r={r} V/E={row['final_V']}/{row['final_E']}")

valid=[x for x in records if x["admissible"]]
passed=sum(bool(x["success"]) for x in valid)
print(f"\nHeadline Yamada recovery: {passed}/{len(valid)} = {100*passed/len(valid):.2f}%")
for x in valid:
    if not x["success"]:
        print("FAILURE:",x["case"],x["transform"],"N=",x["resolution"],"r=",x["radius_vox"],x["error"] or "Yamada mismatch")


In [ ]:
res=RESOLUTIONS; rad=TUBE_RADII_VOX
H=np.full((len(rad),len(res)),np.nan)
for i,r in enumerate(rad):
    for j,N in enumerate(res):
        g=[x for x in records if x["admissible"] and x["radius_vox"]==r and x["resolution"]==N]
        if g:H[i,j]=np.mean([bool(x["success"]) for x in g])
fig,ax=plt.subplots(figsize=(8,3.5))
im=ax.imshow(H,vmin=0,vmax=1,origin="lower",aspect="auto")
ax.set_xticks(range(len(res)),res); ax.set_yticks(range(len(rad)),rad)
ax.set_xlabel("voxel resolution N"); ax.set_ylabel("tube radius [voxels]")
ax.set_title("Yamada recovery rate"); fig.colorbar(im,ax=ax,label="recovery fraction")
plt.show()

names=[c.name for c in CASES]
rates=[]
for n in names:
    g=[x for x in records if x["admissible"] and x["case"]==n]
    rates.append(np.mean([bool(x["success"]) for x in g]) if g else np.nan)
fig,ax=plt.subplots(figsize=(10,4)); ax.bar(range(len(names)),rates); ax.set_ylim(0,1.05)
ax.set_xticks(range(len(names)),names,rotation=60,ha="right"); ax.set_ylabel("Yamada recovery fraction")
plt.tight_layout(); plt.show()


### Interpretation

Only `admissible=True` cases enter the headline recovery fraction. A rejected point means the requested continuous tube is already too thick relative to the conservative clearance guard. An admissible Yamada mismatch is therefore a genuine end-to-end failure caused after a topology-preserving centerline deformation.

The checkpoint file makes the \(N=150\)–\(300\) sweep resumable. Delete it to restart from scratch.
